# Choosing a Strategy

In this notebook we take control of *how* the LLM does the work.

A **strategy** in NOOA is the recipe for how a single method gets run. It decides what the model sees, what tools it can reach, and how it produces its answer. Crucially, you pick a strategy **per method** — different methods on the same class can (and often should) run in very different ways.

To make this concrete, we'll give our barista two jobs that look **nothing alike** — one is a quick vibe read, the other a careful menu pick — and pick a different strategy for each:

- **`read_customer`** — someone walks in and says a single sentence. The barista just needs to make **a quick judgment call** about what they probably want.
- **`recommend_from_menu`** — someone asks for *"something warm and calming, no caffeine."* Now the barista has to actually *look at the menu*, filter, reason, and pick.

NOOA encodes these two shapes with two different **strategies**. (From now on, **strategy** == a decorator on the method that shapes what the LLM sees when that method runs). 

- **`PredictStrategy`** — one prompt, one structured answer. Great for `read_customer`.
- **`CodeActStrategy`** (the default) — a Python REPL where the model can call helpers on `self` before returning. Great for `recommend_from_menu`.

Same class, same LLM, two very different runtime behaviors — chosen method-by-method.

## Prerequisites

Run the install cell below before the setup cell.

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, set your key in Colab's Secrets (🔑 in the left sidebar) as `API_KEY`. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) - those need no API key, just an `api_base`.


In [ ]:
!pip install nooa


## Setup

NOOA works with any LiteLLM-supported model - hosted or local. Pick one below. For hosted providers, set your key in Colab's Secrets (🔑 in the sidebar) as `API_KEY`. Local providers such as Ollama and vLLM do not need a key, just an `api_base`.


In [ ]:
from nooa.unifiedllm.registry import get_llm_client

# In Colab: click the 🔑 icon in the left sidebar, add a secret called API_KEY,
# and enable "Notebook access". Then:
from google.colab import userdata
api_key = userdata.get("API_KEY")

# Locally: `export API_KEY=...` in your shell, then uncomment:
# import os
# api_key = os.environ["API_KEY"]

# model = get_llm_client("claude-haiku-4-5", api_key=api_key)                          # Anthropic
model = get_llm_client("gpt-5.5", api_key=api_key)                                     # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")   # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1") # vLLM (local, no key)


## The Menu

Before we write the agent, let's define the data structures we'll need. There are two:

- `CustomerIntent` — what the barista privately concludes about a customer after hearing one sentence. We use `Literal[...]` so the model has to pick from a fixed vocabulary of tags, plus a confidence score for downstream code to react to. This will be the output shape for our PredictStrategy.
- `DrinkRecommendation` — what the barista returns when actually recommending a drink. This will be the output of our CodeActStrategy.

In [53]:
# Standard imports we'll use across the notebook.
from typing import Literal
from pydantic import BaseModel, Field


# What the barista privately concludes about a customer from one sentence.
# We use Literal[...] so the model is forced to pick from a fixed set of tags,
# and a confidence score so downstream code can react to uncertainty.
class CustomerIntent(BaseModel):
    need: Literal["needs_energy", "avoids_caffeine", "wants_comfort", "in_a_hurry", "wants_treat"]
    confidence: float = Field(ge=0, le=1)
    note: str = Field(description="A brief private note from the barista.")


# What the barista returns when actually recommending a drink.
class DrinkRecommendation(BaseModel):
    drink: str
    reason: str


Then we sketch a small hand-written menu the barista will work from. Each drink has a caffeine amount, a prep time, tags, and a short natural-language note.


In [54]:
# A hand-written menu with notes.
MENU = [
    {
        "drink": "espresso",
        "caffeine_mg": 80,
        "prep_minutes": 2,
        "tags": ["strong", "quick", "classic"],
        "notes": "short, intense, and not sweet",
    },
    {
        "drink": "cappuccino",
        "caffeine_mg": 80,
        "prep_minutes": 4,
        "tags": ["coffee", "milk", "comfort"],
        "notes": "warm, balanced, and familiar",
    },
    {
        "drink": "flat white",
        "caffeine_mg": 120,
        "prep_minutes": 4,
        "tags": ["strong", "milk", "smooth"],
        "notes": "stronger than a cappuccino, but still smooth",
    },
    {
        "drink": "hot chocolate",
        "caffeine_mg": 5,
        "prep_minutes": 5,
        "tags": ["sweet", "comfort", "no_coffee"],
        "notes": "cozy and sweet with almost no caffeine",
    },
    {
        "drink": "chamomile tea",
        "caffeine_mg": 0,
        "prep_minutes": 3,
        "tags": ["calm", "decaf", "gentle"],
        "notes": "soft, herbal, and good late in the day",
    },
]


## Predict Strategy: One Prompt, One Structured Answer

Reading a customer's opening line is a bounded judgment: one short sentence in, one tag from a fixed list out. No menu to inspect, no computation to do — no reason for the model to write code or iterate.

**`PredictStrategy`** is designed for exactly this shape. One prompt goes out, one schema-validated object comes back. No REPL, no tool loop.

You attach it to a specific method with `@strategy(PredictStrategy())`. Everything else on the class keeps the default (CodeAct).


In [55]:
from nooa import Agent, strategy
from nooa.strategies import PredictStrategy


class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""

    # @strategy(...) overrides the default CodeActStrategy for this one method.
    # PredictStrategy means: send one prompt, get one structured answer back,
    # no REPL and no tool loop.
    @strategy(PredictStrategy())
    async def read_customer(self, opening_line: str) -> CustomerIntent:
        """Classify what the customer most likely needs from their opening line.

        Pick the single best-fitting need and include a concise private note.
        """
        ...  


Try it on a handful of opening lines.


In [56]:
agent = BaristaAgent()

# A little cross-section of the kinds of customers the cafe sees.
openers = [
    "I slept badly and need something strong.",
    "I am late for a meeting. What is fast?",
]

# Each call is a single LLM round-trip. The return value is a real
# CustomerIntent object — not a string we have to parse.
for line in openers:
    intent = await agent.read_customer(line)
    print(f"Customer: {line!r}")
    print(f"  need: {intent.need}  confidence={intent.confidence:.2f}")
    print(f"  note: {intent.note}")
    print()


Customer: 'I slept badly and need something strong.'
  need: needs_energy  confidence=0.95
  note: Customer sounds tired and is asking for a strong caffeinated boost.

Customer: 'I am late for a meeting. What is fast?'
  need: in_a_hurry  confidence=0.98
  note: Customer explicitly says they are late and wants the quickest option.

Real `CustomerIntent` objects come back, not strings. Remember: the return type annotation isn't decoration — it's a schema the framework enforces.


### 🥷 Under the hood

Every strategy is really just a *particular way of shaping the prompt* sent to the LLM. To see what that looks like for `PredictStrategy`, we can sneak up on the model and use `print_prompt`, which renders the exact outgoing prompt for a specific call.

In [57]:
from nooa import print_prompt

# Peek at exactly what the LLM will see for one specific call.
await print_prompt(agent.read_customer, opening_line="I love coffee, but I need to sleep tonight.")

=== SYSTEM PROMPT  [BaristaAgent] ===

<system_prompt expr="self._resolve_system_prompt()">
You are a friendly barista at a small neighborhood cafe.
</system_prompt>

<self expr="doc(type(self))">
class BaristaAgent:
    """You are a friendly barista at a small neighborhood cafe."""

    async def read_customer(self, opening_line: str) -> CustomerIntent:
        """
        Classify what the customer most likely needs from their opening line.
        
        Pick the single best-fitting need and include a concise private note.
        """
## Referenced Types
class CustomerIntent(BaseModel):
    need: Literal[needs_energy, avoids_caffeine, wants_comfort, in_a_hurry, wants_treat]
    confidence: float  # [≥0, ≤1]
    note: str  # A brief private note from the barista.
</self>

=== TASK PROMPT  [BaristaAgent.read_customer] ===

# Your task
Classify what the customer most likely needs from their opening line.

Pick the single best-fitting need and include a concise private note.

## Input parameters:
opening_line = 'I love coffee, but I need to sleep tonight.'

Perform the task and return the result directly.

Notice what's *not* in that prompt:

- No `<execution_context>` block advertising a Python REPL.
- No tool-loop instructions, no `execute_python`, no `return_result`.

The model gets the task, the argument value, and the output schema — and nothing else, because nothing else is needed. `PredictStrategy` keeps the prompt as minimal as possible: exactly the information required to produce one structured answer.

## CodeAct: When the Method Has to Actually Do Work

A customer says *"something warm and calming, no caffeine, and I don't want to wait long."* The barista has to:

- Look at the menu.
- Check whether the cafe is in the middle of a rush.
- Filter for low caffeine and quick prep.
- Read the tags and notes and pick something that fits.
- Explain the choice.

This is more than a single one-shot LLM call. What we want is a Python REPL where the model can run `hits = self.find_drinks(...)`, look at `hits`, filter further, and then decide.

That's exactly what **`CodeActStrategy`** does. With CodeAct, the model gets a persistent Python session, sees the agent's public methods, and calls them like any other Python code.

First, a few deterministic helpers on the class. These are just *plain Python* methods — in NOOA, they're automatically available to the agent as tools. We'll keep each one tiny (a couple of lines plus a `print`) so the interesting part is watching the model *stitch several of them together*, not any one of them.

In [58]:
import sys

class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""

    def __init__(self, menu: list[dict], hour: int):
        super().__init__()
        self.menu = menu
        self.hour = hour

    # --- Deterministic helpers. Plain Python. Each one logs when called
    # so you can see the model reaching for it. We write to `sys.__stderr__`
    # (the *original* stderr) rather than `sys.stderr`, because CodeAct wraps
    # the model's code in `redirect_stdout` / `redirect_stderr` to capture output
    # for the tool response — that would otherwise swallow these traces.
    # `sys.__stderr__` bypasses the redirect and prints to the notebook. ---

    def is_rush_hour(self) -> bool:
        """True during the 8–10am morning rush; prep speed matters more then."""
        rush = 8 <= self.hour < 10
        print(f"[is_rush_hour] hour={self.hour} → {rush}", file=sys.__stderr__)
        return rush

    def quick_drinks(self, max_minutes: int = 3) -> list[dict]:
        """Return menu drinks that take at most `max_minutes` to prepare."""
        hits = [d for d in self.menu if d["prep_minutes"] <= max_minutes]
        print(f"[quick_drinks] ≤{max_minutes}min → {[d['drink'] for d in hits]}", file=sys.__stderr__)
        return hits

    def find_drinks(self, query: str) -> list[dict]:
        """Return menu drinks whose name, tags, or notes mention a word from `query`."""
        words = [w for w in query.lower().split() if len(w) > 3]
        hits = [d for d in self.menu if any(w in (d["drink"] + " " + d["notes"] + " " + " ".join(d["tags"])).lower() for w in words)]
        print(f"[find_drinks] {query!r} → {[d['drink'] for d in hits]}", file=sys.__stderr__)
        return hits

    # --- The Predict method from before, unchanged. ---
    @strategy(PredictStrategy())
    async def read_customer(self, opening_line: str) -> CustomerIntent:
        """Classify what the customer most likely needs from their opening line."""
        ...

This is the same agent as before with some deterministic helpers in pure python.

Now the CodeAct method. No body — just an ellipsis. The docstring names the helpers we want the model to reach for.


In [59]:
class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""

    def __init__(self, menu: list[dict], hour: int):
        super().__init__()
        self.menu = menu
        self.hour = hour

    # --- Deterministic helpers (same as above). ---

    def is_rush_hour(self) -> bool:
        """True during the 8–10am morning rush; prep speed matters more then."""
        rush = 8 <= self.hour < 10
        print(f"[is_rush_hour] hour={self.hour} → {rush}", file=sys.__stderr__)
        return rush

    def quick_drinks(self, max_minutes: int = 3) -> list[dict]:
        """Return menu drinks that take at most `max_minutes` to prepare."""
        hits = [d for d in self.menu if d["prep_minutes"] <= max_minutes]
        print(f"[quick_drinks] ≤{max_minutes}min → {[d['drink'] for d in hits]}", file=sys.__stderr__)
        return hits

    def find_drinks(self, query: str) -> list[dict]:
        """Return menu drinks whose name, tags, or notes mention a word from `query`."""
        words = [w for w in query.lower().split() if len(w) > 3]
        hits = [d for d in self.menu if any(w in (d["drink"] + " " + d["notes"] + " " + " ".join(d["tags"])).lower() for w in words)]
        print(f"[find_drinks] {query!r} → {[d['drink'] for d in hits]}", file=sys.__stderr__)
        return hits

    # --- Predict method: bounded judgment, no REPL needed. ---
    @strategy(PredictStrategy())
    async def read_customer(self, opening_line: str) -> CustomerIntent:
        """Classify what the customer most likely needs from their opening line."""
        ...

    # --- CodeAct method (default strategy, no decorator).
    # The docstring names the helpers the model should reach for. ---
    async def recommend_from_menu(self, customer_wants: str) -> DrinkRecommendation:
        """Recommend one drink from the menu.

        Use self.find_drinks to inspect plausible candidates. If self.is_rush_hour(),
        prefer options from self.quick_drinks(). Pick a drink that fits the request
        and explain the choice in one or two sentences.
        """
        ...

Set the clock to 8am — right in the middle of the morning rush — and ask for something warm and fast.

In [60]:
agent = BaristaAgent(MENU, hour=8)

# Under the hood, this call opens a CodeAct session: the model gets a Python
# REPL, sees the helpers on `self` via doc(self), calls the ones it needs,
# and returns a DrinkRecommendation. Watch the [helper] prints to see which
# ones the model reaches for.
rec = await agent.recommend_from_menu("something warm and comforting, and quick — I'm running late")

print(f"\nRecommended: {rec.drink}")
print(f"Why: {rec.reason}")

Recommended: chamomile tea
Why: I’d go with chamomile tea: it’s warm, gentle, and ready in about 3 minutes, so it fits the comforting vibe without slowing you down during the morning rush.

[find_drinks] 'warm comforting comfort quick running late' → ['espresso', 'cappuccino', 'hot chocolate', 'chamomile tea']
[quick_drinks] ≤3min → ['espresso', 'chamomile tea']
[is_rush_hour] hour=8 → True


### 🥷 Under the hood

Let's sneak up on the model again — this time to see the *very different* prompt `CodeActStrategy` produces for the same class.

In [61]:
await print_prompt(agent.recommend_from_menu, customer_wants="something warm and comforting, and quick — I'm running late")

=== SYSTEM PROMPT  [BaristaAgent] ===

<system_prompt expr="self._resolve_system_prompt()">
You are a friendly barista at a small neighborhood cafe.
</system_prompt>

<strategy_prompt>
## Strategy

Jupyter-like Python session. Parameters pre-loaded as locals; state persists across cells. Use `await` directly, `print`/`pprint` to debug, `doc(obj)` to inspect types. You MUST call a tool each turn — **plain-text responses do NOT end the session**. To finish, call `return_result(value)`. Repeated text-only responses will abort the run with an error.

**Your two tools:**
- `execute_python(code)` — run a code cell
- `return_result(value)` — submit your final answer (also callable from inside `execute_python`)

## When to use which tool

Use `return_result(...)` directly for simple answers determinable from the inputs alone (yes/no, one field, a single lookup).

Use `execute_python(...)` for lists/batches, arithmetic, multi-step computation, transforms, or iteration. Always iterate in code — never construct large arrays by hand.

For language tasks (classification, extraction, interpretation), use LLM reasoning — answer directly via `return_result`, or delegate to a `@strategy(PredictStrategy())` standalone function (see below). Don't keyword-match or regex.

## Returning computed results

After computing in code, call `return_result(variable)` **from within** `execute_python()`. This passes the variable directly. Do NOT re-type computed values in a separate `return_result` tool call.

## Helpers

Define helpers at the top of the cell and call them by name. Existing methods on `self` are usable via `await self.method(...)`. Helpers persist as REPL locals across cells in this session.

```python
def normalize(x):
    return x.strip().lower()

cleaned = [normalize(v) for v in values]
```

## Fan-out generation

For per-item LLM work over a list, decorate a standalone async function with `@strategy(PredictStrategy())` and an ellipsis body. `asyncio.gather` runs the calls in parallel.

```python
@strategy(PredictStrategy())
async def detect_language(message: str) -> str:
    """Return the ISO 639-1 language code for {message} (e.g. 'en', 'fr', 'de', 'ja')."""
    ...

codes = await asyncio.gather(*(detect_language(m) for m in messages))
return_result(codes)
```

For iterative sub-tasks that need code execution, use `@strategy(CodeActStrategy())`. The sub-task must be strictly simpler than the current call to avoid infinite recursion.

## Restrictions (will throw)

- `eval`, `exec`, `compile`, `__import__`, `input`, `breakpoint`
- `globals`, `locals`, `vars`, `asyncio.run`, `loop.run_until_complete`
- Attaching callables to the agent: `self.foo = fn`, `setattr(self, 'foo', fn)`, `type(self).foo = fn`
</strategy_prompt>

<execution_context>
## Execution Context

These names are already in scope inside `execute_python()` (state persists across cells) — call them, don't re-import or re-define. Use `doc(name)` to inspect any type or function in detail.

```python
import sys
from nooa import Agent, PredictStrategy, print_prompt, strategy
from nooa.unifiedllm import get_llm_client
from pydantic import BaseModel, Field
from typing import Literal

class BaristaAgent: ...
class CustomerIntent: ...
class DrinkRecommendation: ...
```
Also in scope: exit, get_ipython, open, quit.
Always available without import: `self`, `print()`, `pprint()`, `doc()`, `return_result()`, plus stdlib `asyncio` and `typing`.
</execution_context>

<self expr="doc(type(self))">
class BaristaAgent:
    """You are a friendly barista at a small neighborhood cafe."""

    menu: menu
    hour: hour

    def is_rush_hour(self) -> bool:
        """True during the 8–10am morning rush; prep speed matters more then."""
    def quick_drinks(self, max_minutes: int = 3) -> list[dict]:
        """Return menu drinks that take at most `max_minutes` to prepare."""
    def find_drinks(self, query: str) -> list[dict]:
        """Return menu drinks whose name, tags, or notes mention a word f

Compare this to the Predict prompt above. This one is *much* bigger — and every extra block is earning its keep:

- **`<strategy_prompt>`** — the rulebook for how to act in a REPL: which tools exist (`execute_python`, `return_result`), when to use each, and how to finish a run.
- **`<execution_context>`** — the imports, types, and helpers already in scope inside `execute_python()`. The model doesn't have to guess what's importable — it can just call it.
- **`<self>`** — auto-generated docs of the agent's public methods, rendered from `doc(type(self))`. This is how the model discovers `is_rush_hour`, `quick_drinks`, and `find_drinks` without any tool registration.

`PredictStrategy` stripped the prompt down to the essentials because the task was a one-shot judgment. `CodeActStrategy` hands the model a workshop because the task actually needs one. Same class, same LLM — the strategy decides how much room the model gets to move around.

## When to Use Which

| The method looks like… | Reach for | Because |
|---|---|---|
| Classification, extraction, routing, single structured judgment | `PredictStrategy` | One call, schema-validated output, no Python execution |
| Search, computation, helper calls, external tools, multi-step reasoning | `CodeActStrategy` (default) | Persistent REPL, method calls, iteration, validated return |
| Both shapes on the same class | Mix them freely | Strategy is per method, not per agent |

Rule of thumb: **if the method can answer directly from its arguments, start with Predict. If it needs to inspect or act on live Python state, use CodeAct.**


## Recap

- Strategy is a per-method choice, not a per-agent choice.
- `PredictStrategy` is for bounded judgments — classification, extraction, single-shot structured answers.
- `CodeActStrategy` (default) is for methods that need Python — helpers, iteration, intermediate state, external tools.
- Both strategies return validated Python objects.
- A CodeAct method can `await` a Predict method as one of its steps.

Notebook 3 sticks with CodeAct but scales the state up dramatically — a full public-domain book catalog with tens of thousands of rows. Once state is that big, the interesting question is what the model actually gets to *see* of it.


## Bonus: Composing Strategies

From the caller's point of view, a Predict method and a CodeAct method are both just async methods on `self`. So a CodeAct method can `await` a Predict method as one step in its body.

Here's `serve_customer`: read the customer first (Predict), then recommend a drink (CodeAct).


In [62]:
class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""

    def __init__(self, menu: list[dict], hour: int):
        super().__init__()
        self.menu = menu
        self.hour = hour

    # --- Same helpers as before. ---

    def is_rush_hour(self) -> bool:
        """True during the 8–10am morning rush; prep speed matters more then."""
        rush = 8 <= self.hour < 10
        print(f"[is_rush_hour] hour={self.hour} → {rush}", file=sys.__stderr__)
        return rush

    def quick_drinks(self, max_minutes: int = 3) -> list[dict]:
        """Return menu drinks that take at most `max_minutes` to prepare."""
        hits = [d for d in self.menu if d["prep_minutes"] <= max_minutes]
        print(f"[quick_drinks] ≤{max_minutes}min → {[d['drink'] for d in hits]}", file=sys.__stderr__)
        return hits

    def find_drinks(self, query: str) -> list[dict]:
        """Return menu drinks whose name, tags, or notes mention a word from `query`."""
        words = [w for w in query.lower().split() if len(w) > 3]
        hits = [d for d in self.menu if any(w in (d["drink"] + " " + d["notes"] + " " + " ".join(d["tags"])).lower() for w in words)]
        print(f"[find_drinks] {query!r} → {[d['drink'] for d in hits]}", file=sys.__stderr__)
        return hits

    # --- The Predict method is unchanged. ---
    @strategy(PredictStrategy())
    async def read_customer(self, opening_line: str) -> CustomerIntent:
        """Classify what the customer most likely needs from their opening line."""
        ...

    # --- New: a CodeAct method that composes both. The docstring tells the
    # model to await self.read_customer first, then use the helpers. ---
    async def serve_customer(self, opening_line: str) -> DrinkRecommendation:
        """Read what the customer needs, then recommend one drink from the menu.

        Start by awaiting self.read_customer. Then use self.find_drinks,
        self.quick_drinks, and self.is_rush_hour to choose a drink.
        """
        ...

In [63]:
agent = BaristaAgent(MENU, hour=8)

# serve_customer runs as CodeAct. Inside its REPL, the model will
# `await self.read_customer(...)` — which is itself a Predict call —
# then call the deterministic helpers to pick a drink.
rec = await agent.serve_customer("Ugh, my train leaves in ten minutes and I still need my morning fix.")

print(f"\nRecommended: {rec.drink}")
print(f"Why: {rec.reason}")

Recommended: espresso
Why: You’re short on time and need a morning fix, so I’d make you an espresso: it’s caffeinated, ready in about 2 minutes, and ideal during the rush.

[is_rush_hour] hour=8 → True
[quick_drinks] ≤3min → ['espresso', 'chamomile tea']
[find_drinks] 'energy strong morning caffeine quick' → ['espresso', 'flat white', 'hot chocolate']
